In [ ]:
from Hand import Hand
from Controller import Controller

import math
import time
from enum import Enum, auto
import platform

# utils
from utils import *
from draw_utils import *

import cv2
import matplotlib.pyplot as plt
import numpy as np
np.set_printoptions(precision=4)


# hand landmark: https://google.github.io/mediapipe/solutions/hands.html
import mediapipe as mp 
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

In [ ]:
def preprocess_img(image):
  h, w = image.shape[:2]
  # resize
  if h < w:
    img = cv2.resize(image, (DESIRED_WIDTH, math.floor(h/(w/DESIRED_WIDTH))))
  else:
    img = cv2.resize(image, (math.floor(w/(h/DESIRED_HEIGHT)), DESIRED_HEIGHT))

  return img


In [ ]:
def landmarks2vec(hand_landmarks):
  vecs = np.empty((21, 3))
  for landmark_idx in mp_hands.HandLandmark:
    vecs[landmark_idx] = np.array([
      hand_landmarks.landmark[landmark_idx].x,
      hand_landmarks.landmark[landmark_idx].y,
      hand_landmarks.landmark[landmark_idx].z,
    ])
  return vecs

In [ ]:
class FINGER_STATE(Enum):
  BENT = 1
  STRAIGHT = 0
  UNKNOWN = auto()

class FINGER(Enum):
  THUMB = 0
  INDEX = 1
  MIDDLE = 2
  RING = 3
  PINKY = 4


def get_finger_state(landmarks):
  # TODO: detect thumb state
  res = np.array([FINGER_STATE.UNKNOWN for i in range(5)])
  
  # finger_idx
  # thumb: 1~4, index: 5~8, middle: 9~12, ring: 13~16, pinky: 17~20
  for finger_idx in range(1, 21, 4):
    tip = landmarks[finger_idx+3, :, 0]
    dip = landmarks[finger_idx+2, :, 0]
    pip = landmarks[finger_idx+1, :, 0]
    mcp = landmarks[finger_idx, :, 0]

    accumulated_angle = (np.pi - angle_between_vectors(pip - mcp, pip - dip)) + (np.pi - angle_between_vectors(dip - pip, dip - tip))
    
    if finger_idx == 0:
      # NOTE: special case -> thumb
      res[finger_idx] = FINGER_STATE.BENT if accumulated_angle > (np.pi * 0.25) else FINGER_STATE.STRAIGHT
    else :
      res[(finger_idx-1) // 4] = FINGER_STATE.BENT if accumulated_angle > (np.pi * 0.4) else FINGER_STATE.STRAIGHT

  return res

In [ ]:
DEBUG_MOUSE = False
DEBUG_KEYBOARD = False 

DESIRED_HEIGHT = 720
DESIRED_WIDTH = 720

In [ ]:
controller = Controller()
controller.print_system_info()

if DEBUG_MOUSE:
  mouse_listener = controller.get_mouse_listener()
  mouse_listener.start()
  mouse_listener.wait()
if DEBUG_KEYBOARD:
  keyboard_listener = controller.get_keyboard_listener()
  keyboard_listener.start()
  keyboard_listener.wait()

# fps
s_time = time.time()
frame_cnt = 0
prev_frame_cnt = 0
prev_timestamp = time.time()

# gesture
click_queue = np.ones(4)

# DEBUG: params
tmax = -100

is_first = True

cap = cv2.VideoCapture(0)
with mp_hands.Hands(
    min_detection_confidence=0.75,
    min_tracking_confidence=0.7) as hands:
  while cap.isOpened():
    success, raw_image = cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      continue

    # Flip the image horizontally for a later selfie-view display, and convert the BGR image to RGB.
    image = cv2.cvtColor(cv2.flip(preprocess_img(raw_image), 1), cv2.COLOR_BGR2RGB)
    # To improve performance, optionally mark the image as not writeable to pass by reference.
    # NOTE: see no different
    image.flags.writeable = False
    results = hands.process(image)
    
    image_hight, image_width, _ = image.shape
    # Draw the hand annotations on the image.
    # NOTE: see no different
    # image.flags.writeable = True
    annotated_image = image


    if is_first == True:
      hand = Hand(
        image_width=image_width, 
        image_hight=image_hight, 
        should_saves=[]
      )
      hand.build()
      # TODO: set init landmarks as first frame
      is_first = False


    # predict
    hand.predict()

    # update 
    z_existence = 1 if bool(results.multi_hand_landmarks) else 0

    z_handedness = None
    z_landmarks = None
    if z_existence == 1:
      # handedness
      raw_handedness = results.multi_handedness[0]
      z_handedness = .5 + (.5 if raw_handedness.classification[0].index == 1 else -.5) * raw_handedness.classification[0].score 

      # landmarks
      z_landmarks = landmarks2vec(results.multi_hand_landmarks[0])

    hand.update(z_existence=z_existence, z_handedness=z_handedness, z_landmarks=z_landmarks)

    # save
    # WARN: only save when getting z. otherwise, `Saver.z` and `Saver.x_post` would contain `None`
    if z_existence == 1:
      hand.save()


    existence = hand.existence_x
    handedness = hand.handedness_x
    landmarks = hand.landmarks_x

    
    if existence[0] > .5:

      # Gesture
      finger_states = get_finger_state(landmarks)
      # FIXME: adjust distance of diff depth to the same scale
      ti = np.linalg.norm(landmarks[4, :, 0] - landmarks[6, :, 0]) > 23.
      im = np.linalg.norm(landmarks[8, :, 0] - landmarks[12, :, 0]) > 40.
      mr = np.linalg.norm(landmarks[12, :, 0] - landmarks[16, :, 0]) > 40.
      rp = np.linalg.norm(landmarks[16, :, 0] - landmarks[20, :, 0]) > 40.
      
      # mouse click / drag
      ## prev
      # DEV: `ti` cannot separate idle and click clearly 
      # if ti:
      if finger_states[0] == FINGER_STATE.STRAIGHT:
        if np.all(click_queue == FINGER_STATE.BENT.value):
          controller.mouse_release('left')
          draw_click_drag(annotated_image, landmarks, is_click=False, is_drag=False)
          print('release!')

        elif np.all(click_queue[:-1] == FINGER_STATE.BENT.value) or np.all(click_queue[:-2] == FINGER_STATE.BENT.value) or np.all(click_queue[:-3] == FINGER_STATE.BENT.value):
          # FUTURE: current condition cannot support "double click"
          controller.mouse_click('left')
          draw_click_drag(annotated_image, landmarks, is_click=True, is_drag=False)
          print('click!')

      # DEV: `ti` cannot separate idle and click clearly
      # elif not ti: 
      elif finger_states[0] == FINGER_STATE.BENT:
        if np.all(click_queue[:-1] == FINGER_STATE.BENT.value):
          if click_queue[-1] == FINGER_STATE.STRAIGHT.value:
            controller.mouse_press('left')
            print('press!')
          draw_click_drag(annotated_image, landmarks, is_click=False, is_drag=True)

      ## update
      if finger_states[0].value != FINGER_STATE.UNKNOWN:
        click_queue = np.roll(click_queue, 1)
        # DEV: `ti` cannot separate idel and click clearly
        # click_queue[0] = FINGER_STATE.BENT.value if ti else FINGER_STATE.STRAIGHT.value
        click_queue[0] = finger_states[0].value


      # move mouse
      # DEV:
      if im:
      # if np.all(finger_states[[2,3]] == FINGER_STATE.BENT) and (
            # (np.all(np.abs(landmarks[0, 0:2, 1]) < 10.))
            # or 
            # (finger_states[1] == FINGER_STATE.STRAIGHT)
          # ):

        dx = landmarks[8, 0, 1]
        dy = landmarks[8, 1, 1]
        controller.mouse_move(dx, dy)

      # scroll vertically
      SCALE_FACTOR = .07

      ## scroll down
      # DEV:
      if mr: 
      # if np.all(finger_states[[3,4]] == FINGER_STATE.BENT) and \
            # np.all(finger_states[[1,2]] == FINGER_STATE.STRAIGHT):
        scroll_y = np.clip(landmarks[8, 1, 1]*SCALE_FACTOR, -controller.MAX_SCROLL_SPEED, 0)
        controller.scroll(0, scroll_y)
        # annotate
        cv2.arrowedLine(annotated_image, landmarks[8, 0:2, 0].astype(int), (landmarks[8, 0:2, 0] + [0, -scroll_y*10]).astype(int), color=(255, 50, 50), thickness=3)

      ## scroll up 
      if rp:
      # if np.all(finger_states[[4]] == FINGER_STATE.BENT) and \
            # np.all(finger_states[[1,2,3]] == FINGER_STATE.STRAIGHT):
        scroll_y = np.clip(-landmarks[8, 1, 1]*SCALE_FACTOR, 0, controller.MAX_SCROLL_SPEED)
        controller.scroll(0, scroll_y)
        # annotate
        cv2.arrowedLine(annotated_image, landmarks[8, 0:2, 0].astype(int), (landmarks[8, 0:2, 0] + [0, -scroll_y*10]).astype(int), color=(255, 50, 50), thickness=3)


      # Switch Desktop (4 l/r)
      if np.all(finger_states[[1,2,3,4]] == FINGER_STATE.STRAIGHT) and \
        finger_states[0] == FINGER_STATE.BENT:
        if landmarks[8, 0, 1] > 150:
          controller.switch_desktop_left()
        elif landmarks[8, 0, 1] < -150:
          controller.switch_desktop_right()
        elif landmarks[8, 1, 1] < -150:
          controller.show_control_center()
        elif landmarks[8, 1, 1] > 150:
          controller.show_app_expose()
          

      # Control Center (4 u)
      # App Expose (4 d)
      # LaunchPad (3 in)
      # Show Desktop (3 out)


      # Draw 
      # measurement landmarks
      # mp_drawing.draw_landmarks(annotated_image, raw_landmarks, mp_hands.HAND_CONNECTIONS)
      # draw_landmarks(annotated_image, np.stack([z_landmarks, np.zeros(z_landmarks.shape)], axis=2), (255, 50, 50, 0.5))

      # Draw normal landmark
      draw_landmarks(annotated_image, landmarks)

      # FUTURE: Draw projected landmarks
      # inv_landmarks = np.stack([
        # shift_back_to_origin_coordinate(landmarks[:, :, 0], rotation_matrix, delta_origin), 
        # shift_back_to_origin_coordinate(landmarks[:, :, 1], rotation_matrix, delta_origin)], axis=2)
      # draw_landmarks(annotated_image, inv_landmarks)
      # FUTURE: Draw projection axis
      # annotate_3axis(annotated_image, rotation_matrix, delta_origin)      

      # Draw handedness
      # draw_handedness(annotated_image, handedness)

      # Draw finger states
      # draw_finger_state(annotated_image, handedness, finger_states)
      
      # Draw move click / drag
      # is_click = np.all(click_queue[:-1] == FINGER_STATE.BENT.value) or np.all(click_queue[:-2] == FINGER_STATE.BENT.value) or np.all(click_queue[:-3] == FINGER_STATE.BENT.value)
      # is_drag = np.all(click_queue== FINGER_STATE.BENT.value)
      # draw_click_drag(annotated_image, landmarks, is_click, is_drag)
      

      # DEBUG: output 
      # if time.time() - s_time > 1:
        # tmax = np.max([tmax, np.abs(landmarks_s.z[-1].reshape(21,3)[8, 2] - landmarks_s.z[-2].reshape(21,3)[8, 2]) / dt])
      # tmax = np.max([tmax, np.abs(landmarks_f.x.reshape(21,3,2)[8,2,1])])
        # cv2.putText(annotated_image, 
            #  f'{landmarks_f.x.reshape(21,3,2)[8,1,1] :.2f}',
            #  f'{time.time() % 20}',
            #  org=(int(image_width*.01), int(image_hight*.15)), # bottomLeftCornerOfText
            #  fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
            #  fontScale=.8,
            #  color=(255, 255, 255),
            #  lineType=2)

    else: 
      # TODO: reduce to default position and uncertainty
      pass

    frame_cnt += 1
    now_timestamp = time.time()
    if now_timestamp - prev_timestamp >= 1:
      prev_timestamp, prev_frame_cnt = now_timestamp, frame_cnt
      frame_cnt = 0
    cv2.putText(annotated_image, f'{prev_frame_cnt}', org=(image_width - 30, 20), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.5, color=(100, 255, 100), lineType=2)


    annotated_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
    cv2.imshow('Hands', annotated_image)
    if cv2.waitKey(20) & 0xFF == 27:
      if DEBUG_MOUSE:
      # FIXME: `Listener.stop()` failed to stop mouse listerner
        mouse_listener.stop()
      if DEBUG_KEYBOARD:
        keyboard_listener.stop()
      break

cap.release()
if platform.system() == 'Windows':
  cv2.destroyAllWindows()

# get Saver
dt = hand._dt
existence_s = hand.existence_s
handedness_s = hand.handedness_s
landmarks_s = hand.landmarks_s

## Testing

In [ ]:
xs = np.asarray(landmarks_s.x_post).reshape(-1,21,3,2)
ts = np.arange(xs.shape[0]) * 1/15

plt.figure(figsize=(30, 8))
start = 0
end = None

def dist(a, b, std=False):
  a = np.array(a) 
  b = np.array(b) 
  d = np.linalg.norm(a[:, :] - b[:, :], axis=1)
  return d
  # return (d - np.mean(d)) / np.std(d)
  
# Base Line
plt.plot(ts[start:end], [0]*len(ts[start:end]), color='black')
# plt.plot(ts[start:end], [23]*len(ts[start:end]), color='black')
# plt.plot(ts[start:end], [30]*len(ts[start:end]), color='black')

# Distance between Tips
# plt.plot(ts[start:end], dist(xs[start:end, 4, :, 0], xs[start:end, 6, :, 0]), marker='.', label='ti_1')
# plt.plot(ts[start:end], dist(xs[start:end, 8, :, 0], xs[start:end, 12, :, 0]), marker='.', label='im_1')
# plt.plot(ts[start:end], dist(xs[start:end, 12, :, 0], xs[start:end, 16, :, 0]), marker='.', label='mr_1')
# plt.plot(ts[start:end], dist(xs[start:end, 16, :, 0], xs[start:end, 20, :, 0]), marker='.', label='rp_1')

# Tips Move Speed
plt.plot(ts[start:end], xs[start:end, 16, 0, 1], marker='.', label='xs')
plt.plot(ts[start:end], xs[start:end, 16, 1, 1], marker='.', label='ys')

plt.legend(loc="best")

In [ ]:
zs = np.asanyarray(landmarks_s.z).reshape(-1, 21, 3)#[:-1]
prior = np.asanyarray(landmarks_s.x_prior).reshape(-1, 21, 3, 2)#[1:]
post = np.asanyarray(landmarks_s.x_post).reshape(-1, 21, 3, 2)
ts = np.arange(zs.shape[0]) * dt
print(ts.shape)

In [ ]:
plt.figure(figsize=(15, 15))
# yv: 100
idx = 8
xyz = 1
start = 0
end = -1

plt.subplot(611)
plt.plot(ts[start:end], zs[start:end, idx, 0], marker='.', label='z')
plt.plot(ts[start:end], prior[start:end, idx, 0, 0], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 0, 0], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('x pos')

plt.subplot(612)
plt.plot(ts[start:end], prior[start:end, idx, 0, 1], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 0, 1], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('x vel')

plt.subplot(613)
plt.plot(ts[start:end], zs[start:end, idx, 1], marker='.', label='z')
plt.plot(ts[start:end], prior[start:end, idx, 1, 0], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 1, 0], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('y pos')

plt.subplot(614)
plt.plot(ts[start:end], prior[start:end, idx, 1, 1], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 1, 1], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('y vel')

plt.subplot(615)
plt.plot(ts[start:end], zs[start:end, idx, 2], marker='.', label='z')
plt.plot(ts[start:end], prior[start:end, idx, 2, 0], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 2, 0], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('z pos')

plt.subplot(616)
plt.plot(ts[start:end], prior[start:end, idx, 2, 1], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 2, 1], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('z vel')

In [ ]:
zs_std = np.std(zs, axis=0)
zs_mean = np.mean(zs, axis=0)
prior_std = np.std(prior, axis=0)
prior_mean = np.mean(prior, axis=0)
post_std = np.std(post, axis=0)
post_mean = np.mean(post, axis=0)
print(zs_std.shape, prior_std.shape, post_std.shape)

In [ ]:
plt.figure(figsize=(12, 6))
# plt.errorbar(np.arange(21)+.0, np.zeros(21), yerr=zs_std[:, 0], linestyle='None', fmt='o', label='x')
# plt.errorbar(np.arange(21)+.1, np.zeros(21), yerr=zs_std[:, 1], linestyle='None', fmt='o', label='y')
# plt.errorbar(np.arange(21)+.2, np.zeros(21), yerr=zs_std[:, 2], linestyle='None', fmt='o', label='z')
plt.errorbar(np.arange(21)+.3, np.zeros(21), yerr=prior_std[:, 0, 1], linestyle='None', fmt='o', label='prior-x')
plt.errorbar(np.arange(21)+.4, np.zeros(21), yerr=prior_std[:, 1, 1], linestyle='None', fmt='o', label='prior-y')
plt.errorbar(np.arange(21)+.5, np.zeros(21), yerr=prior_std[:, 2, 1], linestyle='None', fmt='o', label='prior-z')
# plt.errorbar(np.arange(21)+.6, np.zeros(21), yerr=post_std[:, 0, 1], linestyle='None', fmt='o', label='post-x')
# plt.errorbar(np.arange(21)+.7, np.zeros(21), yerr=post_std[:, 1, 1], linestyle='None', fmt='o', label='post-y')
# plt.errorbar(np.arange(21)+.8, np.zeros(21), yerr=post_std[:, 2, 1], linestyle='None', fmt='o', label='post-z')
plt.xticks(np.arange(21))
plt.legend()
# print(np.std(zs_std[:, :], axis=0))
print(prior_std[:, :, 1].shape)

In [ ]:
plt.figure(figsize=(12,6))
np.set_printoptions(suppress=True)
print(np.round(prior_std[:, :, 1], 3))
plt.plot(np.arange(21), prior_std[:, 0, 1], marker='.', label='x')
plt.plot(np.arange(21), prior_std[:, 1, 1], marker='.', label='y')
plt.plot(np.arange(21), prior_std[:, 2, 1], marker='.', label='z')
plt.xticks(np.arange(21))
plt.legend()
plt.show()

In [ ]:
max_v = np.max(prior_std[:,:,1], axis=0)

# print(max_v)
ratio = np.round(prior_std[:,:,1]/max_v, 4)
print(ratio)

plt.figure(figsize=(10,6))
plt.plot(np.arange(21), ratio[:, 0], marker='.', label='x')
plt.plot(np.arange(21), ratio[:, 1], marker='.', label='y')
plt.plot(np.arange(21), ratio[:, 2], marker='.', label='z')
plt.xticks(np.arange(21))
plt.legend()
plt.show()